# Sprint 7 — Generador de Datos Sintéticos
## Proyecto: Productividad Asesores de Negocios

---

### Objetivo

Generar una muestra sintética anonimizada de ~400 asesores que preserve
las distribuciones estadísticas del dataset real para usar en el dashboard.

### Criterios de calidad

- Distribución de TARGET_B balanceada (100 por cuartil)
- Rangos de variables numéricas dentro del rango real
- Correlaciones entre variables preservadas aproximadamente
- Ningún valor real del dataset original — solo estadísticos agregados

### ⚠️ Importante

> Los datos sintéticos NO son datos reales. Solo sirven para demostrar
> el funcionamiento del dashboard. Las predicciones sobre datos sintéticos
> no tienen validez de negocio.


## 1. Configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

INTERIM   = Path('../data/interim')
PROCESSED = Path('../data/processed')
SAMPLE    = Path('../data/sample')
SAMPLE.mkdir(parents=True, exist_ok=True)

# Cargar dataset real a nivel asesor
df_real = pd.read_parquet(INTERIM / 'target_b_full.parquet')
label_encoder = joblib.load(PROCESSED / 'label_encoder.joblib')
CLASS_NAMES = list(label_encoder.classes_)

print(f'Dataset real: {df_real.shape}')
print(f'Clases: {CLASS_NAMES}')
print(f'Distribucion real TARGET_B:')
print(df_real['TARGET_B'].value_counts())


Dataset real: (2004, 40)
Clases: ['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO']
Distribucion real TARGET_B:
TARGET_B
Q1_BAJO          501
Q2_MEDIO_BAJO    501
Q3_MEDIO_ALTO    501
Q4_ALTO          501
Name: count, dtype: int64


## 2. Generador sintético

Estrategia: para cada cuartil de TARGET_B, calculamos los estadísticos
(media, std, min, max) de las variables numéricas en el dataset real
y muestreamos desde distribuciones normales truncadas.

Las variables categóricas se muestrean respetando sus frecuencias reales por cuartil.


In [2]:
# Variables numéricas a generar
NUM_COLS = [
    'GRUPOS', 'CLIENTES', 'PRESTAMO', 'CLIENTES_PRESTAMO',
    'TASA_PROM', 'INCREMENTO_CARTERA', 'CLIENTES_NUEVOS',
    'DESEMBOLSO_CLIENTES_NUEVOS', 'N_SEMANAS_OBS', 'EDAD',
]

# Variables categóricas a generar
CAT_COLS = [
    'REGION', 'RANGO_CICLO', 'DIA_PAGO', 'PUESTO',
    'PRODUCTO', 'AREA', 'TIPO_BAJA', 'MOTIVO_BAJA',
    'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'SEXO', 'ESTADO_CIVIL',
]

N_POR_CLASE = 100  # 100 asesores por cuartil = 400 total

def generar_sinteticos(df_real, target_col, num_cols, cat_cols, n_por_clase, seed=42):
    '''
    Genera datos sintéticos preservando distribuciones por clase.
    
    Para numéricos: muestrea de distribución normal truncada
    usando media y std calculados sobre el dataset real por clase.
    Para categóricos: muestrea respetando frecuencias reales por clase.
    '''
    np.random.seed(seed)
    frames = []

    for clase in df_real[target_col].dropna().unique():
        df_clase = df_real[df_real[target_col] == clase]
        n_real   = len(df_clase)
        sintetico = {}

        # Variables numéricas
        for col in num_cols:
            if col not in df_clase.columns:
                continue
            serie = df_clase[col].dropna()
            if len(serie) == 0:
                sintetico[col] = np.zeros(n_por_clase)
                continue
            media  = serie.mean()
            std    = serie.std()
            v_min  = serie.min()
            v_max  = serie.max()
            if std < 1e-6:
                sintetico[col] = np.full(n_por_clase, media)
            else:
                # Distribución normal truncada entre min y max reales
                a = (v_min - media) / std
                b = (v_max - media) / std
                sintetico[col] = stats.truncnorm.rvs(
                    a, b, loc=media, scale=std, size=n_por_clase
                )

        # Variables categóricas
        for col in cat_cols:
            if col not in df_clase.columns:
                continue
            frecuencias = df_clase[col].value_counts(normalize=True, dropna=True)
            if len(frecuencias) == 0:
                sintetico[col] = np.full(n_por_clase, 'SD')
            else:
                sintetico[col] = np.random.choice(
                    frecuencias.index,
                    size=n_por_clase,
                    p=frecuencias.values,
                )

        df_s = pd.DataFrame(sintetico)
        df_s[target_col] = str(clase)
        frames.append(df_s)

    return pd.concat(frames, ignore_index=True)


print('Generando datos sintéticos...')
df_sintetico = generar_sinteticos(
    df_real, 'TARGET_B', NUM_COLS, CAT_COLS, N_POR_CLASE
)

print(f'Dataset sintético: {df_sintetico.shape}')
print(f'Distribucion TARGET_B:')
print(df_sintetico['TARGET_B'].value_counts())


Generando datos sintéticos...
Dataset sintético: (400, 23)
Distribucion TARGET_B:
TARGET_B
Q2_MEDIO_BAJO    100
Q1_BAJO          100
Q4_ALTO          100
Q3_MEDIO_ALTO    100
Name: count, dtype: int64


## 3. Validación de calidad del dataset sintético

In [3]:
print('=== VALIDACION DE CALIDAD ===')
print()

# Test 1: rangos numéricos
print('Test 1 — Rangos numéricos (sintético vs real):')
print(f'{"Variable":<30} {"Min real":>10} {"Min sint":>10} {"Max real":>10} {"Max sint":>10}')
print('─' * 65)
ok_rangos = True
for col in NUM_COLS:
    if col not in df_sintetico.columns or col not in df_real.columns:
        continue
    min_r = df_real[col].min()
    max_r = df_real[col].max()
    min_s = df_sintetico[col].min()
    max_s = df_sintetico[col].max()
    # Verificar que el sintético no excede el rango real
    dentro = (min_s >= min_r - abs(min_r)*0.05) and (max_s <= max_r + abs(max_r)*0.05)
    marca = 'OK' if dentro else 'REVISAR'
    if not dentro:
        ok_rangos = False
    print(f'{col:<30} {min_r:>10.2f} {min_s:>10.2f} {max_r:>10.2f} {max_s:>10.2f}  {marca}')

print(f'\nTest 1: {"PASADO" if ok_rangos else "REVISAR"}')

# Test 2: sin valores nulos en columnas críticas
nulos_crit = df_sintetico[NUM_COLS].isna().sum().sum()
print(f'\nTest 2 — Nulos en numéricas: {nulos_crit} ({"PASADO" if nulos_crit == 0 else "FALLO"})')

# Test 3: distribución de TARGET_B balanceada
dist = df_sintetico['TARGET_B'].value_counts()
balanceado = dist.std() < 5
print(f'\nTest 3 — Distribución balanceada: {dict(dist)} ({"PASADO" if balanceado else "REVISAR"})')

# Test 4: medias por clase preservan el orden esperado
# Q4_ALTO debe tener TASA_PROM mayor que Q1_BAJO
media_tasa_q1 = df_sintetico[df_sintetico['TARGET_B']=='Q1_BAJO']['TASA_PROM'].mean()
media_tasa_q4 = df_sintetico[df_sintetico['TARGET_B']=='Q4_ALTO']['TASA_PROM'].mean()
orden_ok = media_tasa_q4 > media_tasa_q1
print(f'\nTest 4 — Orden de medias preservado:')
print(f'  TASA_PROM Q1_BAJO={media_tasa_q1:.1f} < Q4_ALTO={media_tasa_q4:.1f}: {"PASADO" if orden_ok else "REVISAR"}')


=== VALIDACION DE CALIDAD ===

Test 1 — Rangos numéricos (sintético vs real):
Variable                         Min real   Min sint   Max real   Max sint
─────────────────────────────────────────────────────────────────
GRUPOS                               1.00       1.00       2.25       1.26  OK
CLIENTES                             6.00       6.18      18.35      13.19  OK
PRESTAMO                             0.00     119.16  456000.00   94348.36  OK
CLIENTES_PRESTAMO                    0.00       0.00      13.00       7.45  OK
TASA_PROM                           82.56      83.01     355.49     318.79  OK
INCREMENTO_CARTERA              -26290.58  -11403.50  456000.00   88051.28  OK
CLIENTES_NUEVOS                      0.00       0.00      13.00       3.75  OK
DESEMBOLSO_CLIENTES_NUEVOS           0.00       0.38  456000.00   56019.02  OK
N_SEMANAS_OBS                        1.00       1.43     137.00     134.69  OK
EDAD                                 0.00      11.24      58.00      5

## 4. Guardar dataset sintético

In [4]:
# Guardar en data/sample/ — esta carpeta SÍ se versiona en GitHub
output_path = SAMPLE / 'advisors_sample_anon.parquet'
df_sintetico.to_parquet(output_path, index=False)

size_kb = output_path.stat().st_size / 1024
print(f'Dataset sintético guardado:')
print(f'  Ruta    : {output_path}')
print(f'  Tamaño  : {size_kb:.1f} KB')
print(f'  Filas   : {len(df_sintetico):,}')
print(f'  Columnas: {df_sintetico.shape[1]}')
print(f'\nColumnas disponibles:')
print(df_sintetico.columns.tolist())

# Estadísticos de referencia para el dashboard
stats_ref = df_real[NUM_COLS].agg(['mean','std','min','max']).round(2)
stats_ref.to_csv(SAMPLE / 'stats_referencia.csv')
print(f'\nEstadísticos de referencia guardados en data/sample/stats_referencia.csv')


Dataset sintético guardado:
  Ruta    : ..\data\sample\advisors_sample_anon.parquet
  Tamaño  : 51.4 KB
  Filas   : 400
  Columnas: 23

Columnas disponibles:
['GRUPOS', 'CLIENTES', 'PRESTAMO', 'CLIENTES_PRESTAMO', 'TASA_PROM', 'INCREMENTO_CARTERA', 'CLIENTES_NUEVOS', 'DESEMBOLSO_CLIENTES_NUEVOS', 'N_SEMANAS_OBS', 'EDAD', 'REGION', 'RANGO_CICLO', 'DIA_PAGO', 'PUESTO', 'PRODUCTO', 'AREA', 'TIPO_BAJA', 'MOTIVO_BAJA', 'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'SEXO', 'ESTADO_CIVIL', 'TARGET_B']

Estadísticos de referencia guardados en data/sample/stats_referencia.csv
